# 24 · Capstone — Recommender System

**Part 3 · Capstones** · *Prerequisites: Parts 1 & 2* · *Both books, one system*

The recommender is where the two halves of this course physically meet: an **event log**
(nb 11/12) feeds **batch training** (nb 10) of embeddings, a **two-stage serving path**
(nb 18) ranks under a latency budget, a **policy layer** (nb 13) applies business rules,
and the system's own outputs become its future training data — nb 19's feedback loop,
now load-bearing and dangerous. We build the whole spine.

**Step 1 — Requirements.** Personalized home-page recommendations; p99 **< 150 ms**;
catalog and tastes change weekly (retraining is a given, nb 20); new items must get
discovered (cold start); business wants engagement *with* diversity (two objectives —
decouple, nb 13).

**Step 2 — Workload & architecture.** The defining scale fact: scoring **every** item
per request with a real model is impossible (10⁸ items × ms-scale model = hours). Hence
the universal industry shape — **two stages**: a cheap, vectorizable **candidate
generator** cuts the catalog to a few hundred; an expensive **ranker** orders those.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

from sysdes import viz

viz.use_style()
rng = np.random.default_rng(seed=24)

fig, ax = viz.system_diagram(
    nodes=[
        {"id": "app", "x": 0, "y": 1.2, "label": "clients", "kind": "client"},
        {"id": "serve", "x": 2.3, "y": 1.2, "label": "rec service"},
        {"id": "cand", "x": 4.7, "y": 2.2, "label": "candidate gen\n(embeddings)", "kind": "service"},
        {"id": "rank", "x": 4.7, "y": 1.2, "label": "ranker (GBM)\n+ policy layer", "kind": "service"},
        {"id": "online", "x": 4.7, "y": 0.1, "label": "online features\n(nb 16)", "kind": "cache"},
        {"id": "log", "x": 2.3, "y": -0.9, "label": "event log\n(impressions, clicks)", "kind": "queue"},
        {"id": "batch", "x": 4.9, "y": -1.9, "label": "nightly batch:\nMF training +\nfeature build", "kind": "service"},
        {"id": "reg", "x": 7.3, "y": -0.9, "label": "registry +\nmonitors\n(nb 19-21)", "kind": "store"},
    ],
    edges=[
        {"src": "app", "dst": "serve", "label": "get_recs", "bidir": True},
        {"src": "serve", "dst": "cand", "label": "top-200"},
        {"src": "cand", "dst": "rank", "label": ""},
        {"src": "rank", "dst": "serve", "label": "top-10", "curve": 0.3},
        {"src": "serve", "dst": "online", "bidir": True, "style": "dashed"},
        {"src": "serve", "dst": "log", "label": "impressions + clicks\n(+ propensities!)"},
        {"src": "log", "dst": "batch", "label": ""},
        {"src": "batch", "dst": "cand", "label": "new embeddings", "style": "dashed", "curve": -0.4},
        {"src": "batch", "dst": "reg", "label": "gate", "style": "dashed"},
    ],
    title="One log in the middle; everything else is derived from it (nb 12)",
)
plt.show()

## Step 3a — A world with real taste structure

Users belong to latent **taste groups**; items to **categories**; clicks follow taste
affinity plus a popularity pull. (We keep the ground truth so every stage is gradeable.)

In [ ]:
N_USERS, N_ITEMS, N_GROUPS, DIM = 3_000, 1_500, 8, 6

group_vecs = rng.normal(0, 1, (N_GROUPS, DIM))
user_group = rng.integers(0, N_GROUPS, N_USERS)
item_group = rng.integers(0, N_GROUPS, N_ITEMS)
user_taste = group_vecs[user_group] + rng.normal(0, 0.4, (N_USERS, DIM))
item_attr = group_vecs[item_group] + rng.normal(0, 0.4, (N_ITEMS, DIM))
item_pop = rng.lognormal(0, 1, N_ITEMS)
pop_z = (np.log(item_pop) - np.log(item_pop).mean()) / np.log(item_pop).std()

def click_prob(users: np.ndarray, items: np.ndarray) -> np.ndarray:
    """The hidden truth: taste match + a popularity pull."""
    affinity = (user_taste[users] * item_attr[items]).sum(axis=1)
    return 1 / (1 + np.exp(-(0.85 * affinity + 0.4 * pop_z[items] - 2.4)))

# Interaction history: 80 random impressions per user; clicks by the truth.
hist_users = np.repeat(np.arange(N_USERS), 80)
hist_items = rng.integers(0, N_ITEMS, len(hist_users))
hist_clicks = rng.random(len(hist_users)) < click_prob(hist_users, hist_items)
pos_u, pos_i = hist_users[hist_clicks], hist_items[hist_clicks]
print(f"history: {len(hist_users):,} impressions, {hist_clicks.sum():,} clicks "
      f"(CTR {hist_clicks.mean():.1%})")

## Step 3b — Candidate generation: matrix factorization

Learn a `DIM_MF`-dimensional embedding per user and item such that clicked pairs score
high — trained only on the click log (implicit feedback: observed positives + sampled
negatives), by mini-batch SGD on a logistic loss. At serving, "candidates for user u" =
top-k items by dot product — one vectorized matmul over the whole catalog:

In [ ]:
DIM_MF, EPOCHS, LR, NEG_PER_POS = 16, 20, 0.08, 2

U = rng.normal(0, 0.1, (N_USERS, DIM_MF))
V = rng.normal(0, 0.1, (N_ITEMS, DIM_MF))

def sgd_epoch(users, items, labels):
    order = rng.permutation(len(users))
    for start in range(0, len(order), 4096):
        b = order[start:start + 4096]
        u, i, y = users[b], items[b], labels[b]
        pred = 1 / (1 + np.exp(-(U[u] * V[i]).sum(axis=1)))
        g = (pred - y)[:, None]                       # dLoss/dscore
        gU, gV = g * V[i], g * U[u]
        np.add.at(U, u, -LR * (gU + 0.01 * U[u]))     # L2-regularized updates
        np.add.at(V, i, -LR * (gV + 0.01 * V[i]))

for epoch in range(EPOCHS):
    neg_i = rng.integers(0, N_ITEMS, NEG_PER_POS * len(pos_u))   # sampled negatives
    ep_users = np.concatenate([pos_u, np.repeat(pos_u, NEG_PER_POS)])
    ep_items = np.concatenate([pos_i, neg_i])
    ep_labels = np.concatenate([np.ones(len(pos_u)),
                                np.zeros(NEG_PER_POS * len(pos_u))])
    sgd_epoch(ep_users, ep_items, ep_labels.astype(float))

# Grade it as a CANDIDATE model: recall@200 on fresh ground-truth clicks -
# does the true next click make it into the top-200 shortlist?
def recall_at_k(score_items_for, k=200, n_eval=600):
    hits = 0
    eval_users = rng.choice(N_USERS, n_eval, replace=False)
    for u in eval_users:
        probe = rng.integers(0, N_ITEMS, 60)          # fresh impressions
        clicked = probe[rng.random(60) < click_prob(np.full(60, u), probe)]
        if len(clicked) == 0:
            continue
        target = clicked[0]
        hits += target in score_items_for(u)[:k]
    return hits / n_eval

mf_candidates = lambda u: np.argsort(-(U[u] @ V.T))
pop_candidates = lambda u: np.argsort(-item_pop)      # nb 17's baseline rung
recall_mf = recall_at_k(mf_candidates)
recall_pop = recall_at_k(pop_candidates)
print(f"recall@200 - popularity baseline: {recall_pop:.2f}, "
      f"matrix factorization: {recall_mf:.2f}")
assert recall_mf > 1.35 * recall_pop, \
    "the embeddings find 35%+ more of the next clicks than popularity alone"

(At real scale the `U[u] @ V.T` over 10⁸ items becomes an **approximate nearest
neighbor** index — HNSW/IVF — trading a few points of recall for 1000× speed; exercise 2.
The *architecture* is unchanged.)

## Step 3c — Ranking: a model over the shortlist

The ranker sees only ~200 candidates, so it can afford real features (nb 16): the MF
score, popularity, and the user's **historical affinity for the item's category** (an
online-store feature — updated by the stream, nb 11):

In [ ]:
# User->category affinity from the click history (a batch feature, nb 14).
cat_affinity = np.zeros((N_USERS, N_GROUPS))
np.add.at(cat_affinity, (pos_u, item_group[pos_i]), 1.0)
cat_affinity /= cat_affinity.sum(axis=1, keepdims=True).clip(1)

def features_for(users, items):
    return np.c_[(U[users] * V[items]).sum(axis=1),          # MF score
                 pop_z[items],                                # popularity
                 cat_affinity[users, item_group[items]]]      # category affinity

# Train the ranker on logged impressions (features must be point-in-time
# honest: everything above is computable from the history alone, nb 16).
X_rank = features_for(hist_users, hist_items)
ranker = HistGradientBoostingClassifier(random_state=0).fit(X_rank, hist_clicks)

# Evaluate both stages on fresh impressions:
ev_u = rng.integers(0, N_USERS, 20_000)
ev_i = rng.integers(0, N_ITEMS, 20_000)
ev_y = rng.random(20_000) < click_prob(ev_u, ev_i)
auc_mf = roc_auc_score(ev_y, (U[ev_u] * V[ev_i]).sum(axis=1))
auc_rank = roc_auc_score(ev_y, ranker.predict_proba(features_for(ev_u, ev_i))[:, 1])
print(f"AUC - MF score alone: {auc_mf:.3f}, ranker with features: {auc_rank:.3f}")
assert auc_rank > auc_mf + 0.02, \
    "stage two earns its latency: features the embedding can't carry"

## Step 3d — The serving path, assembled (with its policy layer)

Candidates → rank → **policy**: cap categories for diversity (a business rule that must
live *outside* the weights, nb 13 §3), reserve one **exploration slot** (nb 19's
antidote, priced in), cache per-user lists with a short TTL:

In [ ]:
SLATE, CAND_K, MAX_PER_CAT, EXPLORE_SLOT = 10, 200, 3, 9

def recommend(u: int, explore: bool = True) -> np.ndarray:
    cands = np.argpartition(-(U[u] @ V.T), CAND_K)[:CAND_K]      # stage 1: cheap
    scores = ranker.predict_proba(features_for(np.full(CAND_K, u), cands))[:, 1]
    order = cands[np.argsort(-scores)]                           # stage 2: precise
    slate, per_cat = [], np.zeros(N_GROUPS, int)                 # stage 3: policy
    for item in order:
        if per_cat[item_group[item]] < MAX_PER_CAT:
            slate.append(int(item))
            per_cat[item_group[item]] += 1
        if len(slate) == SLATE:
            break
    if explore:                                                  # the truth serum
        slate[EXPLORE_SLOT] = int(rng.integers(0, N_ITEMS))
    return np.array(slate)

t0 = time.perf_counter()
slates = [recommend(int(u)) for u in rng.choice(N_USERS, 60, replace=False)]
ms = (time.perf_counter() - t0) / 60 * 1e3
counts = np.array([np.bincount(item_group[s[:EXPLORE_SLOT]], minlength=N_GROUPS).max()
                   for s in slates])
print(f"serving: {ms:.1f} ms/request (budget 150); "
      f"max items from one category (excl. explore slot): {counts.max()}")
assert counts.max() <= MAX_PER_CAT, "the diversity rule is enforced in the policy layer"
assert ms < 100, "two-stage keeps the expensive model off the full catalog"

## Step 4 — Closing the loop *safely*: cold start needs the explore slot

A brand-new item has no embedding (it was never in the training log): the candidate
stage scores it near zero, so it is never shown, so it never earns clicks, so the next
retrain still knows nothing — nb 19's selective-labels trap, verbatim. The exploration
slot is the escape hatch. Prove it:

In [ ]:
# A new item arrives, genuinely GREAT for taste group 3:
new_item = N_ITEMS                                # id one past the catalog
item_group_x = np.append(item_group, 3)
item_attr_x = np.vstack([item_attr, group_vecs[3]])
V_x = np.vstack([V, np.zeros(DIM_MF)])            # cold: zero embedding

def serve_round(explore: bool, rounds: int = 12_000) -> int:
    """Serve slates; count clicks the NEW item earns (= its future training data)."""
    clicks = 0
    users = rng.integers(0, N_USERS, rounds)
    for u in users:
        scores = U[u] @ V_x.T
        shown = list(np.argsort(-scores)[:SLATE])   # true top-10 (ranker elided for speed)
        if explore:
            shown[EXPLORE_SLOT] = int(rng.integers(0, N_ITEMS + 1))
        if new_item in shown:
            affinity = user_taste[u] @ item_attr_x[new_item]
            clicks += rng.random() < 1 / (1 + np.exp(-(0.55 * affinity - 0.8)))
    return clicks

clicks_without = serve_round(explore=False)
clicks_with = serve_round(explore=True)
print(f"clicks earned by the new item over 12,000 slates - "
      f"no exploration: {clicks_without}, with explore slot: {clicks_with}")
assert clicks_without == 0, "cold item + pure exploitation = permanent invisibility"
assert clicks_with > 0, "the explore slot bought the training data the next retrain needs"

Alongside the slot, the impression event we log carries the **propensity** (why/how
this item was shown — rank, score, explore-or-not): that field is what lets training
correct for position and exposure bias later (nb 15 ex. 3, nb 19 §4). Logging it costs
one column; not logging it costs the ability to ever de-bias.

## Steps 4½ & 5 — Failure analysis & evolution (the checklist)

- **Embedding refresh skew**: nightly batch ships new `V`; if the ranker's `mf_score`
  feature is computed with *new* V while the ranker was trained on *old* V's score
  distribution — train/serve skew (nb 16). Ship `(U, V, ranker)` as one versioned unit
  through the registry gate (nb 21), never piecemeal.
- **Batch job fails** (nb 21's orchestrator alarm): serve yesterday's embeddings —
  degraded, not down. Recs age gracefully; that's a virtue of derived data.
- **Cold users**: no embedding → fall back to popularity + context features (the
  baseline rung never really retires).
- **Cache staleness**: a user clicks item X and it's *still on their cached slate* —
  invalidate-on-event via the log (nb 12), or short TTLs; embarrassment, not incident.
- **Monitoring** (nb 19): CTR by slice, score-PSI per stage, candidate-overlap between
  model versions, share-of-impressions concentration (the Gini from nb 19 §4 — your
  feedback-loop alarm).
- **Evolution** (nb 20): retrains ride the trigger pipeline; new rankers climb shadow →
  canary → A/B on *engagement per user*, with diversity and latency as guardrails.

## Exercises

1. **Negative sampling.** Our MF sampled negatives uniformly. What bias does that
   create against popular items, what does sampling negatives ∝ popularity fix, and
   what new failure does it risk? (Connect to nb 15's sampling chapter.)
2. **ANN trade.** At 10⁸ items, exact top-200 is off the table. An HNSW index returns
   ~0.95 recall@200 in 2 ms. Where exactly does the lost 0.05 recall go (which items
   are hardest for ANN), and why does the two-stage design blunt the damage?
3. **Position bias.** Training on logged clicks, slot 1 gets 10× slot 10's attention.
   Show how the ranker learns "whatever we used to rank first is good", and how the
   logged propensity field + inverse-propensity weighting (or randomized interleaving)
   breaks the tautology.
4. **Objective decoupling.** Product wants engagement, editorial wants quality,
   sales wants promoted items. Design the policy-layer scoring (nb 13 §3) so each
   knob is tunable without retraining — and name the guardrail metric each owner
   watches in the A/B.
5. **The matched pair.** Explain concretely what breaks if user embeddings U are
   refreshed nightly but item embeddings V weekly. Which nb 16 concept is this, and
   what does the registry record to prevent it?

### Solutions

**1.** Uniform negatives make popular items *relatively* over-represented as positives
→ the model partly learns popularity, not taste (and unpopular items are "easy"
negatives that teach little). Popularity-proportional negatives force the model to
distinguish taste *within* the popular set — harder, better gradients — but overdone,
it suppresses genuinely popular-because-good items and starves tail items of any
signal. It's importance sampling (nb 15) applied to the loss; tune it like one.

**2.** ANN misses concentrate on items in sparse/boundary regions of embedding space —
which skews against niche and *new* items (compounding cold start — another reason the
explore slot exists). The two-stage design blunts it because stage 1 only needs the
target *somewhere* in 200 (recall), not ranked correctly; a 0.95-recall shortlist costs
at most 5% of ultimately-best items, and the ranker's job is unaffected for the rest.

**3.** Clicks ∝ true appeal × position exposure; training on raw clicks teaches the
model the *product*, and since the old ranker chose positions, the new model inherits
the old one's preferences — a self-licking ranking. With propensity e(position) logged,
weight each example by 1/e (clicks from slot 10 count ~10×) → unbiased appeal
estimates; or serve randomized pairs (interleaving) to measure appeal free of position.
No propensity logged = no retroactive fix — which is why the field ships on day one.

**4.** `final = w_e·engagement + w_q·quality + w_p·promo_boost`, all three scores from
separate models/tables, weights in config (nb 13's α, plural), plus hard floors
(quality_score > q_min) as constraints. Guardrails: engagement owner watches
CTR/session length; editorial watches avg quality of impressed items; sales watches
promo share; *everyone* watches long-term retention (nb 20 ex. 3's slow metric).

**5.** Dot products only mean anything **within one training run's coordinate system**
— U-from-Tuesday against V-from-last-week is a comparison across two arbitrary bases:
scores become noise, quietly (nothing crashes; CTR just sags — the worst failure mode,
nb 19's silence). This is train/serve skew (nb 16) at the artifact level. The registry
records the *pair* (U, V — plus the ranker trained downstream of them) as one versioned
deployable with a shared training-run id; the gate refuses mixed versions (nb 21 ex. 3's
feature-compatibility check).

## Takeaways

- **Two stages** is the load-bearing idea: a cheap vectorized recaller over everything,
  a real model over a shortlist, a *policy layer* over the model — each stage tunable
  and testable alone (baseline rungs, nb 17, at system scale).
- The event log is the spine: features, embeddings, monitoring, and retraining are all
  **derived** from it (nb 12) — and impressions must log *propensities*, because the
  system trains on data it chose to create (nb 19).
- Exploration is not waste; it is the purchase of tomorrow's training data — the only
  cure for cold items and calcified rankings.
- Ship embeddings + ranker as a **matched, gated pair**; serve stale on batch failure;
  fall back to popularity for cold users. Degrade, don't die.

**Next:** [25 · Fraud detection](25-capstone-fraud-detection.ipynb) — the final
capstone: milliseconds, adversaries, and money.